%md
#Rocket Lab
## Arquitetura Medalhão 
#### Camada Silver

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
catalog = "workspace"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

DataFrame[]

In [0]:
# ----- funções auxiliares -----

# Mantém só o registro mais recente de cada chave (pela data de ingestão)
def mais_recente(df, chave):
    janela = Window.partitionBy(chave).orderBy(F.col("ingestion_datetime").desc())
    return df.withColumn("rn", F.row_number().over(janela)).filter("rn = 1").drop("rn")

# conversão segura: texto inválido vira NULL em vez de quebrar o pipeline
def converter(coluna, tipo):
    return F.expr(f"try_cast({coluna} as {tipo})")

# grava a tabela na Silver (overwrite: a Silver é recalculada a cada execução)
def salvar(df, tabela):
    nome = f"{silver_schema}.{tabela}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome)
    print(f"{nome}: {spark.table(nome).count()} linhas")

In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_info").withColumn("id", F.trim("id"))
df = mais_recente(df, "id").filter(F.col("id").isNotNull())

# status: minúsculo e só letras (remove hífens, espaços e ruídos) antes de traduzir
status = F.regexp_replace(F.lower(F.col("status")), "[^a-z]", "")
status_traduzido = (
    F.when(status == "released", "Lançado")
    .when(status == "postproduction", "Pós-Produção")
    .when(status == "inproduction", "Em Produção")
    .when(status == "planned", "Planejado")
    .when(status == "rumored", "Rumores")
    .when(status.isin("canceled", "cancelled"), "Cancelado")
    .otherwise("Não Informado")
)

# data: testa vários formatos; só fica NULL se nenhum funcionar
formatos = ["yyyy-MM-dd", "yyyy-MM-dd HH:mm:ss", "dd/MM/yyyy", "MM/dd/yyyy", "yyyy/MM/dd",
            "dd-MM-yyyy", "dd.MM.yyyy", "yyyyMMdd", "MMMM d, yyyy", "d MMMM yyyy"]
data = F.to_date(F.coalesce(*[F.expr(f"try_to_timestamp(trim(release_date), '{f}')") for f in formatos]))

df_info_filmes = (
    df.withColumn("data_lancamento", data)
    .withColumn("duracao_minutos", converter("runtime", "double").cast("int"))
    .select(
        F.col("id").alias("id_filme"),
        F.trim("title").alias("titulo"),
        F.trim("original_title").alias("titulo_original"),
        "data_lancamento",
        F.year("data_lancamento").alias("ano_lancamento"),
        "duracao_minutos",
        F.trim("original_language").alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        F.col("overview").alias("sinopse"),
        F.col("tagline").alias("frase_divulgacao"),
    )
)

salvar(df_info_filmes, "tb_info_filmes")
display(df_info_filmes.groupBy("status_filme").count())

workspace.silver.tb_info_filmes: 97611 linhas


status_filme,count
Lançado,96261
Em Produção,604
Pós-Produção,697
Planejado,47
Não Informado,2


In [0]:
df = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .withColumn("data_cotacao", F.to_date(F.substring("dataHoraCotacao", 1, 10)))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
)

# 1 cotação por dia (a última do dia)
janela_dia = Window.partitionBy("data_cotacao").orderBy(F.col("dataHoraCotacao").desc())
df = df.withColumn("rn", F.row_number().over(janela_dia)).filter("rn = 1").select("data_cotacao", "cotacao_compra")

# do primeiro dia com cotação até hoje
data_inicio = df.agg(F.min("data_cotacao")).first()[0]
calendario = spark.sql(f"SELECT explode(sequence(to_date('{data_inicio}'), current_date(), interval 1 day)) AS data_cotacao")

#  dia sem cotação recebe o valor do último dia útil
janela_ffill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_cotacao = (
    calendario.join(df, "data_cotacao", "left")
    .withColumn("dia_com_cotacao", F.col("cotacao_compra").isNotNull())
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))
)

salvar(df_cotacao, "tb_cotacao_dolar")
display(df_cotacao.orderBy("data_cotacao"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


workspace.silver.tb_cotacao_dolar: 8 linhas


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra,dia_com_cotacao
2026-09-30,5.1803,true
2026-10-01,5.2073,true
2026-10-02,5.2232,true
2026-10-03,5.2232,false
2026-10-04,5.2232,false
2026-10-05,4.9853,true
2026-10-06,4.9692,true
2026-10-07,4.9929,true


In [0]:
textos_ausentes = ["unknown", "não informado", "nao informado", "n/a", "na", "null", "none", "-", ""]

def limpar_valor(coluna):
    c = F.lower(F.trim(F.col(coluna)))
    # texto de ausência -> NULL
    c = F.when(c.isin(textos_ausentes), None).otherwise(c)
    # remove $, R$, USD, espaços            
    c = F.regexp_replace(c, "[^0-9.,]", "")
    # remove milhar com vírgula                                       
    c = F.regexp_replace(c, ",", "")                                              
    c = F.when(c.rlike(r"^\d{1,3}(\.\d{3})+$"), F.regexp_replace(c, r"\.", "")).otherwise(c)  
    return c

# Cotação mais recente da série
cotacao = spark.table(f"{silver_schema}.tb_cotacao_dolar").orderBy(F.col("data_cotacao").desc()).first()["cotacao_compra"]
print(f"Cotação usada: R$ {cotacao}")

df = spark.table(f"{bronze_schema}.tb_movies_financials").withColumn("id", F.trim("id"))
df = (
    mais_recente(df, "id")
    .filter(F.col("id").isNotNull())
    .withColumn("budget", limpar_valor("budget"))
    .withColumn("revenue", limpar_valor("revenue"))
    .withColumn("orcamento_usd", converter("budget", "decimal(18,2)"))
    .withColumn("receita_usd", converter("revenue", "decimal(18,2)"))
)

# 0 ou negativo = ausente
for c in ["orcamento_usd", "receita_usd"]:
    df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))

taxa = F.lit(cotacao).cast("decimal(10,4)")
tem_valores = F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull()

df_financeiro = (
    df.withColumn("orcamento_brl", (F.col("orcamento_usd") * taxa).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * taxa).cast("decimal(18,2)"))
    # Lucro só é calculado quando os dois valores existem
    .withColumn("lucro_usd", F.when(tem_valores, F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", F.when(tem_valores, F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem: evita divisão por zero (receita já é > 0 ou NULL)
    .withColumn("margem_lucro_percentual",
                F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2).cast("decimal(18,2)"))
    .select(F.col("id").alias("id_filme"), "orcamento_usd", "receita_usd", "orcamento_brl",
            "receita_brl", "lucro_usd", "lucro_brl", "margem_lucro_percentual")
)

salvar(df_financeiro, "tb_financeiro_filmes")
display(df_financeiro.filter(F.col("lucro_usd").isNotNull()).limit(10))

Cotação usada: R$ 4.9929
workspace.silver.tb_financeiro_filmes: 99006 linhas


id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1001724,3.00,8000.00,14.98,39943.20,7997.00,39928.22,99.96
1003578,10000.00,3000000.00,49929.00,14978700.00,2990000.00,14928771.00,99.67
1008042,4500000.00,72600000.00,22468050.00,362484540.00,68100000.00,340016490.00,93.80
1009615,4.00,10000.00,19.97,49929.00,9996.00,49909.03,99.96
1012642,25.00,123.00,124.82,614.13,98.00,489.31,79.67
1014127,500.00,10000.00,2496.45,49929.00,9500.00,47432.55,95.00
1014193,10.00,200.00,49.93,998.58,190.00,948.65,95.00
1024777,10000.00,60000.00,49929.00,299574.00,50000.00,249645.00,83.33
1026837,40000.00,800.00,199716.00,3994.32,-39200.00,-195721.68,-4900.00
1028194,1.00,400.00,4.99,1997.16,399.00,1992.17,99.75


In [0]:
df = spark.table(f"{bronze_schema}.tb_movies_metrics").withColumn("id", F.trim("id"))
df = mais_recente(df, "id").filter(F.col("id").isNotNull())

#padroniza o separador decimal para ponto
pop = F.trim(F.col("popularity"))
pop = (
    F.when(pop.contains(",") & pop.contains("."),
           F.when(F.instr(pop, ",") > F.instr(pop, "."),
                  F.regexp_replace(F.regexp_replace(pop, r"\.", ""), ",", "."))   # 1.234,56
           .otherwise(F.regexp_replace(pop, ",", "")))                            # 1,234.56
    .otherwise(F.regexp_replace(pop, ",", "."))                                   # 12,5
)
df = df.withColumn("popularity", F.regexp_replace(pop, "[^0-9.-]", ""))

df = (
    df.withColumn("popularidade", converter("popularity", "double"))
    .withColumn("nota_media_tmdb", converter("vote_average", "double"))
    .withColumn("qtd_votos_tmdb", converter("vote_count", "double").cast("int"))
    .withColumn("nota_media_imdb", converter("averageRating", "double"))
    .withColumn("qtd_votos_imdb", converter("numVotes", "double").cast("int"))
)

# regras de negócio: notas entre 0 e 10; votos e popularidade não negativos
for c in ["nota_media_tmdb", "nota_media_imdb"]:
    df = df.withColumn(c, F.when(F.col(c).between(0, 10), F.col(c)))
for c in ["popularidade", "qtd_votos_tmdb", "qtd_votos_imdb"]:
    df = df.withColumn(c, F.when(F.col(c) >= 0, F.col(c)))

df_metricas = df.select(F.col("id").alias("id_filme"), "popularidade", "nota_media_tmdb",
                        "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")

salvar(df_metricas, "tb_metricas_engajamento")
display(df_metricas.limit(10))

workspace.silver.tb_metricas_engajamento: 95115 linhas


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236
1000081,102.802,5.142,134,5.1,1841
1000092,4.797,null,null,null,130
1000094,2.645,6.9,28,6.5,null
1000096,0.71,10.0,1,6.3,695
1000099,0.6,0.0,0,8.6,33
1000108,3.231,0.0,0,6.5,157
1000110,1.38,0.0,0,6.9,263


In [0]:
df_avaliacoes = (
    spark.table(f"{bronze_schema}.tb_movies_reviews")
    .withColumn("nota", F.regexp_replace(F.trim("nota"), ",", "."))
    .select(
        F.trim("id").alias("id_filme"),
        F.trim("nome").alias("nome_usuario"),
        converter("nota", "double").alias("nota_usuario"),
        F.col("comentario").alias("comentario_usuario"),
    )
    # Nota fora de 0 a 10 vira NULL
    .withColumn("nota_usuario", F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))
    # Comentário vazio ou só com espaços
    .withColumn("comentario_usuario",
                F.when(F.trim(F.coalesce("comentario_usuario", F.lit(""))) == "", "Sem comentário")
                .otherwise(F.trim("comentario_usuario")))
    # Remove avaliações totalmente duplicadas
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

salvar(df_avaliacoes, "tb_avaliacoes_usuarios")
display(df_avaliacoes.limit(10))

workspace.silver.tb_avaliacoes_usuarios: 32412 linhas


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


In [0]:
df_creditos = spark.table(f"{bronze_schema}.tb_credits_and_tags").withColumn("id", F.trim("id"))
df_creditos = mais_recente(df_creditos, "id").filter(F.col("id").isNotNull())

# Padroniza separadores (; -> ,), quebra em lista e explode: uma linha por valor
def explodir(coluna):
    return (
        df_creditos
        .withColumn("valor", F.explode(F.split(F.regexp_replace(F.col(coluna), ";", ","), ",")))
        .withColumn("valor", F.trim(F.regexp_replace("valor", r'[\[\]"]', "")))
        .select(F.col("id").alias("id_filme"), "valor")
    )

In [0]:
df_generos = (
    explodir("genres")
    # Só nomes curtos com letras: remove vazios, números e textos descritivos deslocados
    .filter(F.col("valor").rlike("^[A-Za-zÀ-ÿ &-]+$") & (F.length("valor") <= 25))
    .select("id_filme", F.initcap(F.lower("valor")).alias("nome_genero"))
    .dropDuplicates()
)

salvar(df_generos, "tb_generos")
display(df_generos.groupBy("nome_genero").count().orderBy(F.desc("count")).limit(15))

workspace.silver.tb_generos: 133237 linhas


nome_genero,count
Drama,30743
Documentary,18850
Comedy,17534
Thriller,9524
Horror,9243
Romance,7056
Action,5586
Crime,4356
Animation,4205
Tv Movie,3709


In [0]:
tipos = {"cast": "Ator", "directors": "Diretor", "writers": "Roteirista", "production_companies": "Produtora"}

df_pessoas = None
for coluna, tipo in tipos.items():
    df_tipo = explodir(coluna).withColumn("tipo_entidade", F.lit(tipo))
    df_pessoas = df_tipo if df_pessoas is None else df_pessoas.unionByName(df_tipo)

df_pessoas = (
    df_pessoas
    # Precisa ter letra, tamanho razoável e não ser só número (resíduos do column shift)
    .filter(F.col("valor").rlike("[A-Za-zÀ-ÿ]") & F.length("valor").between(2, 60))
    .select("id_filme", F.initcap(F.lower("valor")).alias("nome_entidade"), "tipo_entidade")
    .dropDuplicates()
)

salvar(df_pessoas, "tb_pessoas_empresas")
display(df_pessoas.groupBy("tipo_entidade").count())

workspace.silver.tb_pessoas_empresas: 893214 linhas


tipo_entidade,count
Diretor,101462
Roteirista,130584
Produtora,120189
Ator,540979


In [0]:
# verificacao
tabelas = ["tb_info_filmes", "tb_cotacao_dolar", "tb_financeiro_filmes", "tb_metricas_engajamento",
           "tb_avaliacoes_usuarios", "tb_generos", "tb_pessoas_empresas"]

for t in tabelas:
    print(f"{t}: {spark.table(f'{silver_schema}.{t}').count()} linhas")

tb_info_filmes: 97611 linhas
tb_cotacao_dolar: 8 linhas
tb_financeiro_filmes: 99006 linhas
tb_metricas_engajamento: 95115 linhas
tb_avaliacoes_usuarios: 32412 linhas
tb_generos: 133237 linhas
tb_pessoas_empresas: 893214 linhas
